# 梯度通过链式法则逐层相乘

- <font color='red'>在反向传播过程中，梯度通过链式法则逐层相乘</font>
  - 如果这些乘积项大于 1.0，梯度会像滚雪球一样指数级增长，导致权重更新幅度过大，模型无法收敛，甚至 loss 变成 NaN。
  
  
## 一个 3 层网络的梯度爆炸示例

假设有一个简单的 3 层网络，<font color='red'>每层权重矩阵的数值都偏大</font>：

In [ ]:
输入 → [W1] → [W2] → [W3] → 输出

假设每层权重矩阵的"典型数值"（比如谱半径/最大奇异值）都约为 2.0。

## 前向传播：逐层相乘
<font color='red'>前向传播时，信号逐层被放大：</font>

In [ ]:
第1层输出 ≈ W1 × 输入        → 信号 × 2
第2层输出 ≈ W2 × (W1 × 输入) → 信号 × 2 × 2 = 信号 × 4
第3层输出 ≈ W3 × ...         → 信号 × 2 × 2 × 2 = 信号 × 8

<font color='red'>前向传播时信号变大了，但还不至于爆炸。</font>

## 反向传播：梯度也是逐层相乘
反向传播时，梯度按照链式法则从输出层往输入层逐层回传，同样要乘以每层的权重：

In [ ]:
∂L/∂W3 的梯度 ∝ 上游梯度 × 激活值

上游梯度回传到第2层：∂L/∂W2 ∝ ∂L/∂W3 × W3  → 梯度 × 2
继续回传到第1层：  ∂L/∂W1 ∝ ∂L/∂W2 × W2  → 梯度 × 2 × 2 = 梯度 × 4
再往前：          ∂L/∂W0 ∝ ∂L/∂W1 × W1  → 梯度 × 2 × 2 × 2 = 梯度 × 8

<font color='red'>现在看起来还好，才 8 倍。 但关键在于——</font>

## <font color='red'>当网络变深时：指数级增长</font>

<font color='blue'>假设网络有 50 层，每层权重的典型值为 2.0：</font>

In [ ]:
第 1 层的梯度 ∝ 原始梯度 × 2^50

2^50 ≈ 1,125,899,907,000,000（约 10^15）
<font color='red'>这意味着第 1 层收到的梯度比最后一层大了一千万亿倍！</font>

## 具体数值演示
假设 loss 对输出层的梯度初始值为 0.01：

| 回传层数 | 梯度值 | 说明 |
|---|---|---|
| 第 50 层（输出） | 0.01 | 初始梯度 |
| 第 40 层 | 0.01 × 2^10 ≈ 10.24 | 开始变大 |
| 第 30 层 | 0.01 × 2^20 ≈ 10,485 | 已经很大 |
| 第 20 层 | 0.01 × 2^30 ≈ 10,737,418 | 百万级 |
| 第 10 层 | 0.01 × 2^40 ≈ 10,995,116,277 | 百亿级 |
| 第 1 层 | 0.01 × 2^50 ≈ 11,258,999,068,426 | 万亿级 |

此时权重更新：

In [ ]:
W = W - lr × gradient
# W = W - 0.001 × 11,258,999,068,426
# W = W - 11,258,999,068

<font color='red'>权重直接被更新到了一个天文数字，下一轮前向传播的输出也会变成天文数字，loss 直接变成 NaN。</font>

## <font color='blue'>为什么 clip_grad_norm_ 能解决</font>
加入 clip_grad_norm_ 后，假设 max_norm=1.0：

In [ ]:
原始梯度范数 = 11,258,999,068,426
缩放因子 = 1.0 / 11,258,999,068,426 ≈ 8.88 × 10^-14

裁剪后梯度 = 原始梯度 × 缩放因子

<font color='red'>所有梯度被等比缩小，方向不变</font>，幅度被压回安全范围，权重更新就不会失控了。

## 一句话总结
- 链式法则本质是连乘。只要每层的"乘数"大于 1，层数一多，乘积就会指数爆炸。clip_grad_norm_ 就是在爆炸发生前，把整体梯度"按回去"。